# DataTrust — Auditoría de calidad: tabla `listings`

## Contexto

Airbnb Barcelona (Inside Airbnb, snapshot 2026-06-24) es una plataforma real de alquiler turístico. La tabla `listings` contiene 15.293 alojamientos activos en la ciudad, con información del anfitrión, la propiedad, el precio, la licencia turística y las reseñas recibidas.

## Problema

Estos datos se reutilizan para análisis reales (investigación académica sobre vivienda turística, periodismo de datos, seguimiento ciudadano) y Barcelona está actualmente restringiendo las licencias de alquiler turístico en parte por falta de trazabilidad fiable sobre qué alojamientos cumplen la normativa. Pero nadie garantiza que estos datos sean fiables antes de usarlos. Un análisis preliminar ya mostró que **el 53,4% de los alojamientos tiene al menos una incidencia de calidad** dentro de las dimensiones auditadas.

## Objetivo de este notebook

Auditar la tabla `listings` en cinco dimensiones, seleccionadas según su relevancia para el uso posterior de los datos: completitud, duplicados, validez (campo de licencia), consistencia (codificación de caracteres) y coherencia (valores atípicos en precio y estancia mínima). Cada hallazgo se documenta con su evidencia en código. Este notebook es solo de diagnóstico, no se corrige nada todavía, el tratamiento se aborda en un notebook posterior, una vez estén priorizados todos los hallazgos de las tres tablas.

In [1]:
import pandas as pd
import re

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

listings = pd.read_csv("../data/airbnb_bcn/listings.csv", low_memory=False)
print("Filas:", listings.shape[0], "| Columnas:", listings.shape[1])

Filas: 15293 | Columnas: 90


### Función auxiliar: clasificación del campo `license`

Se define aquí, una sola vez, porque se usa en dos puntos del notebook: en la Sección 0 (cifra de apertura) y en la Sección 3 (validez del campo, donde se explica el criterio en detalle). Evita repetir la misma lógica dos veces.

In [3]:
def clasifica_licencia(valor):
    """
    Clasifica un valor del campo license en una de cuatro categorias segun si
    contiene un numero HUTB reconocible, una exencion declarada, esta vacio,
    o es texto libre que no encaja en las anteriores.
    valor: str o NaN - contenido original del campo license
    return: str - una de 'Sin dato', 'Exento (declarado)',
            'Con numero HUTB reconocible', 'Otro / no clasificable'
    """
    if pd.isna(valor):
        return "Sin dato"
    valor = str(valor)
    if "Exempt" in valor:
        return "Exento (declarado)"
    if re.search(r"HUTB[-]?\d{4,}", valor):
        return "Con numero HUTB reconocible"
    return "Otro / no clasificable"

## 0. Magnitud del problema (cifra de apertura)

Antes de entrar dimensión por dimensión, cuantificamos cuántos alojamientos tienen **al menos una** incidencia entre cuatro problemas: licencia no verificable, precio ausente, datos estructurales incompletos (habitaciones/baños/camas), o sin ninguna valoración.

**Por qué estos cuatro y no otros**: son los campos más directamente necesarios para tomar una decisión básica sobre un alojamiento real (¿es de fiar?, ¿me lo puedo permitir?, ¿me cabe mi grupo?, ¿tiene buena reputación?), no una fórmula matemática única, es una selección razonada de campos fundamentales para el uso del dato.

In [4]:
lic_cat = listings["license"].apply(clasifica_licencia)

problema_licencia = lic_cat.isin(["Sin dato", "Otro / no clasificable"])
problema_precio = listings["price"].isna()
problema_estructura = listings[["bedrooms", "bathrooms", "beds"]].isna().any(axis=1)
problema_rating = listings["review_scores_rating"].isna()

al_menos_uno = problema_licencia | problema_precio | problema_estructura | problema_rating

print(f"Con problema de licencia: {problema_licencia.sum()} ({problema_licencia.mean()*100:.1f}%)")
print(f"Con precio nulo: {problema_precio.sum()} ({problema_precio.mean()*100:.1f}%)")
print(f"Con nulos estructurales (bedrooms/bathrooms/beds): {problema_estructura.sum()} ({problema_estructura.mean()*100:.1f}%)")
print(f"Sin valoracion: {problema_rating.sum()} ({problema_rating.mean()*100:.1f}%)")
print()
print(f"Con AL MENOS UNO de estos problemas: {al_menos_uno.sum()} de {len(listings)} ({al_menos_uno.mean()*100:.1f}%)")

Con problema de licencia: 4202 (27.5%)
Con precio nulo: 1938 (12.7%)
Con nulos estructurales (bedrooms/bathrooms/beds): 5016 (32.8%)
Sin valoracion: 3548 (23.2%)

Con AL MENOS UNO de estos problemas: 8164 de 15293 (53.4%)


## 1. Completitud

**Criterio (regla objetiva y reproducible)**: se calcula el porcentaje de valores nulos de cada columna. Se aplican dos umbrales, definidos de antemano y no ajustados después de ver el resultado:
- **100% de nulos → caso crítico**: la columna no aporta ninguna información utilizable.
- **≥10% y <100% de nulos → problema de completitud a documentar**: umbral operativo definido para este proyecto (no un estándar externo verificado), con el objetivo de distinguir ausencias puntuales de niveles de ausencia que pueden afectar al análisis.
- **<10% de nulos**: no se reporta como hallazgo, salvo que la columna sea crítica para el modelo o el negocio.

La causa de un nulo (por ejemplo, si es un campo opcional del perfil o si la plataforma dejó de exponerlo) se considera una hipótesis razonada a partir del patrón de los datos, no un hecho confirmado por documentación oficial de Inside Airbnb, se señala así explícitamente en cada caso.

In [5]:
nulos = listings.isna().sum().sort_values(ascending=False)
nulos_pct = (nulos / len(listings) * 100).round(1)
resumen_nulos = pd.DataFrame({"nulos": nulos, "pct": nulos_pct.astype(str) + "%"})

print("Columnas con 100% de valores nulos:")
print(resumen_nulos[nulos_pct == 100.0])

print()
print("Otras columnas relevantes con nulos parciales:")
cols_clave = ["price", "bedrooms", "bathrooms", "beds", "host_about", "host_location"]
print(resumen_nulos.loc[cols_clave])

Columnas con 100% de valores nulos:
                           nulos     pct
neighborhood_overview      15293  100.0%
host_since                 15293  100.0%
host_response_time         15293  100.0%
host_thumbnail_url         15293  100.0%
host_acceptance_rate       15293  100.0%
host_response_rate         15293  100.0%
host_verifications         15293  100.0%
neighbourhood              15293  100.0%
host_total_listings_count  15293  100.0%
host_neighbourhood         15293  100.0%
calendar_updated           15293  100.0%
instant_bookable           15293  100.0%

Otras columnas relevantes con nulos parciales:
               nulos    pct
price           1938  12.7%
bedrooms        2562  16.8%
bathrooms       3565  23.3%
beds            2447  16.0%
host_about      4859  31.8%
host_location   3098  20.3%


In [6]:
cols_totalmente_vacias = nulos_pct[nulos_pct == 100.0].index.tolist()
cols_host = [c for c in cols_totalmente_vacias if c.startswith("host_")]
cols_otras = [c for c in cols_totalmente_vacias if not c.startswith("host_")]

host_cols_total = [c for c in listings.columns if c.startswith("host_")]
proporcion_host_total = len(host_cols_total) / listings.shape[1]
esperado_por_proporcion = len(cols_totalmente_vacias) * proporcion_host_total

print(f"Columnas 100% vacias que empiezan con 'host_': {len(cols_host)} de {len(cols_totalmente_vacias)} ({len(cols_host)/len(cols_totalmente_vacias)*100:.1f}%)")
print(cols_host)
print()
print("Resto:", cols_otras)
print()
print(f"Columnas 'host_' sobre el TOTAL de columnas del dataset: {len(host_cols_total)} de {listings.shape[1]} ({proporcion_host_total*100:.1f}%)")
print(f"Si la ausencia se repartiera en esa misma proporcion, se esperarian ~{esperado_por_proporcion:.1f} columnas host_ vacias; se observan {len(cols_host)}.")

Columnas 100% vacias que empiezan con 'host_': 8 de 12 (66.7%)
['host_since', 'host_response_time', 'host_thumbnail_url', 'host_acceptance_rate', 'host_response_rate', 'host_verifications', 'host_total_listings_count', 'host_neighbourhood']

Resto: ['neighborhood_overview', 'neighbourhood', 'calendar_updated', 'instant_bookable']

Columnas 'host_' sobre el TOTAL de columnas del dataset: 20 de 90 (22.2%)
Si la ausencia se repartiera en esa misma proporcion, se esperarian ~2.7 columnas host_ vacias; se observan 8.


**Interpretación**: 12 columnas están vacías al 100% (`host_since`, `host_response_rate`, `host_verifications`, `instant_bookable`, `neighborhood_overview`, entre otras). No es un porcentaje pequeño de huecos, es la columna completa.

Una hipótesis plausible es que estos campos hayan dejado de estar disponibles públicamente o hayan cambiado de formato en la fuente, impidiendo su captura en este snapshot (un fenómeno del mismo tipo que SEAT:CODE documentó públicamente para otra fuente de datos de Barcelona, aunque esto no confirma que sea la causa exacta en este caso). Esta explicación no puede confirmarse únicamente a partir del dataset.

**8 de las 12 columnas (66,7%) empiezan con `host_`**, frente al 22,2% que representan las columnas `host_` sobre el total de las 90 columnas del dataset. Si la ausencia se repartiera en esa misma proporción, se esperarían aproximadamente 2,7 columnas `host_` vacías; se observan 8. La ausencia está claramente concentrada en el bloque de variables del anfitrión en comparación con su peso sobre el total de columnas. Esto limita especialmente el análisis de variables relacionadas con la actividad y características del anfitrión, como antigüedad, respuesta y verificaciones.

Los nulos parciales de `price`, `bedrooms`, `bathrooms` y `beds` podrán evaluarse posteriormente para decidir un tratamiento adecuado (imputación, exclusión de ciertos análisis, u otra alternativa), no se asume de antemano que deban rellenarse.

**Nota sobre la cifra de la Sección 0**: estas 12 columnas no se incluyeron en el cálculo de "al menos un problema" de la Sección 0. Si se incluyera una columna vacía en el 100% de las filas, el resultado habría sido automáticamente 100% para todos los alojamientos, una cifra trivial que no mide variación real entre ellos.

### Completitud según tipo de anfitrión (hallazgo secundario)

Hipótesis inicial a comprobar: ¿los anfitriones profesionales tienen perfiles menos completos que los particulares? "Profesional" se define aquí de forma operativa, como **proxy**: `host_listings_count > 50`. No es una confirmación de que se trate de una empresa o agencia en sentido legal/empresarial, es un umbral basado en el volumen de anuncios gestionados. Se comprueba con `host_about` (biografía) y `host_location` (ubicación declarada).

In [7]:
listings["host_profesional"] = listings["host_listings_count"] > 50

for col in ["host_about", "host_location"]:
    tasa_nulos = listings.groupby("host_profesional")[col].apply(lambda s: s.isna().mean() * 100).round(1)
    print(col)
    print(tasa_nulos)
    print()

host_about
host_profesional
False    39.2
True     11.0
Name: host_about, dtype: float64

host_location
host_profesional
False    25.1
True      6.8
Name: host_location, dtype: float64



**Interpretación**: los resultados contradicen la hipótesis inicial. Los anfitriones profesionales presentan **menos** valores ausentes en `host_about` (11,0% frente a 39,2% en particulares) y en `host_location` (6,8% frente a 25,1%), una diferencia considerable.

Una posible explicación es que los perfiles profesionales estén sujetos a procesos operativos más estandarizados, aunque el dataset no permite establecer la causa de esta diferencia, solo describirla.

Este hallazgo se retoma más adelante, en la Sección 3, al comparar con la validez del campo `license` para el mismo grupo de anfitriones.

## 2. Duplicados

**Criterio**: cada alojamiento debe aparecer una sola vez en el dataset. Se comprueba mediante dos controles distintos: unicidad del campo `id`, y ausencia de filas completamente duplicadas (idénticas en todas sus columnas). Son dos comprobaciones necesarias, no intercambiables: podría haber `id` repetido con otras columnas distintas (duplicado parcial) sin que `duplicated()` lo detecte como fila idéntica.

In [8]:
print("Filas totales:", len(listings))
print("IDs unicos:", listings["id"].nunique())
print("Filas duplicadas completas:", listings.duplicated().sum())

Filas totales: 15293
IDs unicos: 15293


Filas duplicadas completas: 0


**Interpretación**: no se detectan problemas de unicidad, los 15.293 alojamientos tienen `id` únicos y no existen filas completamente duplicadas. Esta dimensión está limpia, no todo en el dataset es caótico, y es importante decirlo así de claro: la auditoría no busca "encontrar problemas a toda costa", busca medir con precisión cuáles existen y cuáles no.

## 3. Validez del campo `license` (caso estrella de gobernanza)

**Criterio**: el campo debería contener un número de licencia turística reconocible (formato HUTB) o una exención declarada explícitamente. Definimos cuatro categorías:
- `Con numero HUTB reconocible`: contiene un patrón `HUTB` + dígitos.
- `Exento (declarado)`: contiene la palabra "Exempt".
- `Sin dato`: campo vacío.
- `Otro / no clasificable`: texto libre que no encaja en ninguna de las anteriores.

**Importante**: esta clasificación mide si el campo es *legible/verificable*, no si el alojamiento tiene o no licencia real. Un campo "no clasificable" no equivale a "sin licencia", puede ser una licencia real mal formateada. No hacemos esa afirmación sin una fuente externa que la confirme.

In [9]:
listings["license_categoria"] = listings["license"].apply(clasifica_licencia)

conteo = listings["license_categoria"].value_counts()
porcentaje = (listings["license_categoria"].value_counts(normalize=True) * 100).round(1)
print(pd.DataFrame({"conteo": conteo, "pct": porcentaje.astype(str) + "%"}))

multi_hutb = listings["license"].dropna().apply(lambda v: len(re.findall(r"HUTB[-]?\d{4,}", str(v))))
print()
print("Registros con mas de un numero HUTB concatenado en el mismo campo:", (multi_hutb >= 2).sum())

                             conteo    pct
license_categoria                         
Con numero HUTB reconocible    7167  46.9%
Exento (declarado)             3924  25.7%
Sin dato                       3043  19.9%
Otro / no clasificable         1159   7.6%



Registros con mas de un numero HUTB concatenado en el mismo campo: 5429


In [10]:
cruce = pd.crosstab(listings["license_categoria"], listings["host_profesional"], normalize="index") * 100
cruce.columns = ["% particular", "% profesional"]
print(cruce.round(1))

                             % particular  % profesional
license_categoria                                       
Con numero HUTB reconocible          71.1           28.9
Exento (declarado)                   78.7           21.3
Otro / no clasificable               45.1           54.9
Sin dato                             83.6           16.4


**Interpretación**: 46,9% de los anuncios tiene un número HUTB reconocible, 25,7% declara exención, 19,9% no tiene dato, y 7,6% es texto libre no clasificable. Además, 5.429 registros mezclan dos números de licencia en el mismo campo (nacional + autonómico), con etiquetas HTML sin limpiar: el contenido observado presenta características de texto libre/no estructurado. No podemos afirmar cómo se diseñó originalmente el campo, solo cómo se comporta en este snapshot.

El hallazgo más relevante para negocio: los anfitriones **profesionales (>50 anuncios) están sobrerrepresentados en la categoría "no clasificable"** (54,9%) frente al resto de categorías (16-29%). Esto no demuestra incumplimiento normativo (no es esa la afirmación). Introduce una limitación relevante para cualquier análisis de cumplimiento que utilice este campo tal cual, especialmente al analizar operadores de mayor escala: en este grupo es más frecuente que el contenido de `license` no pueda clasificarse mediante la regla definida. (No se cuantifica aquí qué proporción del inventario total de Airbnb Barcelona controlan estos operadores, ese dato no se ha calculado en este notebook.)

**Nota sobre la dirección de esta lectura**: el cruce anterior normaliza por fila de `license_categoria`, así que el 54,9% responde a "de los anuncios en 'no clasificable', qué porcentaje es profesional". Es la pregunta relevante para este hallazgo (de dónde procede el ruido de validez), pero es una pregunta distinta de "de los profesionales, qué porcentaje tiene licencia no clasificable". Esa segunda pregunta se responde a continuación, como vista complementaria.

In [11]:
cruce_inverso = pd.crosstab(listings["host_profesional"], listings["license_categoria"], normalize="index") * 100
cruce_inverso.index = ["Particular", "Profesional"]
print(cruce_inverso.round(1))

license_categoria  Con numero HUTB reconocible  Exento (declarado)  Otro / no clasificable  Sin dato
Particular                                45.3                27.4                     4.6      22.6
Profesional                               51.2                20.7                    15.7      12.3


**Interpretación (vista complementaria)**: dentro del grupo profesional, el 15,7% tiene licencia "no clasificable", frente al 4,6% en particulares, unas 3,4 veces más. Este es el dato que responde directamente a "¿qué porcentaje de profesionales tiene licencia no clasificable?".

Un matiz importante que esta vista añade: si se suman las dos categorías problemáticas ("Sin dato" + "Otro / no clasificable", la misma definición de `problema_licencia` usada en la Sección 0), el total es similar entre grupos (28,0% profesional vs. 27,2% particular). La diferencia real no está en *cuántos* tienen un problema de licencia en conjunto, sino en *qué tipo* de problema predomina en cada grupo: los particulares concentran su problema en "Sin dato" (22,6%), mientras que los profesionales lo concentran en "Otro / no clasificable" (15,7%), la categoría más difícil de interpretar porque mezcla texto libre, licencias mal formateadas y posibles licencias reales sin reconocer. Esto no contradice el hallazgo original, lo precisa: la sobrerrepresentación profesional es específica del tipo de problema "no clasificable", no del volumen total de problemas de licencia.

**Conexión con la Sección 1**: ya vimos que el grupo de anfitriones profesionales tiene perfiles más completos en `host_about` y `host_location` que los particulares. Aquí se observa lo contrario en la dimensión de validez: ese mismo grupo presenta también una mayor proporción de registros de licencia clasificados como "no clasificable". Es decir, **completitud y validez son dimensiones distintas de la calidad de un dato**: un registro puede estar muy completo en unas variables y presentar problemas de validez en otra. Un perfil más completo no implica automáticamente un registro más fiable en su conjunto.

In [12]:
listings["license_problema"] = listings["license_categoria"].isin(["Sin dato", "Otro / no clasificable"])

por_distrito = listings.groupby("neighbourhood_group_cleansed")["license_problema"].agg(["sum", "count", "mean"])
por_distrito = por_distrito.sort_values("mean", ascending=False)
por_distrito["pct"] = ((por_distrito["mean"] * 100).round(1).astype(str) + "%")
print(por_distrito[["sum", "count", "pct"]])

                               sum  count    pct
neighbourhood_group_cleansed                    
Horta-Guinardó                 143    358  39.9%
Nou Barris                      58    163  35.6%
Ciutat Vella                  1081   3247  33.3%
Sarrià-Sant Gervasi            276    888  31.1%
Sant Andreu                     69    225  30.7%
Gràcia                         416   1366  30.5%
Sant Martí                     417   1436  29.0%
Sants-Montjuïc                 368   1448  25.4%
Les Corts                       82    327  25.1%
Eixample                      1292   5835  22.1%


**Interpretación**: la proporción de anuncios cuyo campo `license` está sin dato o no es clasificable según la regla definida varía por distrito, del 22,1% en Eixample al 39,9% en Horta-Guinardó. Ciutat Vella ocupa el tercer lugar, con un 33,3%. No se afirma una causa de estas diferencias (podrían estar relacionadas con la composición del tipo de anfitrión, la antigüedad de los anuncios u otros factores que este análisis no permite determinar), solo se documenta el patrón geográfico como evidencia adicional para la fase de negocio.

## 4. Consistencia: codificación de caracteres

**Criterio**: los nombres de barrio deberían mostrar correctamente los caracteres propios del catalán/castellano (í, ï, ç, à...).

In [13]:
barrios_con_problema = [b for b in listings["neighbourhood_cleansed"].unique() if "\ufffd" in str(b)]
print("Barrios con caracteres mal codificados:", len(barrios_con_problema))
print(barrios_con_problema[:5])

Barrios con caracteres mal codificados: 0
[]


**Interpretación**: no se detectaron caracteres de reemplazo (�) en los nombres de barrio, por lo que no se observa evidencia de este tipo específico de corrupción de codificación en esta columna (la prueba comprueba la presencia del carácter de reemplazo Unicode, no descarta cualquier otro tipo posible de problema de codificación). No se incluye como hallazgo en el resumen final: se comprobó una dimensión de calidad y el resultado fue satisfactorio.

## 5. Coherencia y valores atípicos

**Criterio importante**: un valor extremo no es automáticamente un error. Distinguimos tres niveles conceptuales:
- **Valor extremo pero posiblemente válido**: requiere contexto de negocio antes de decidir, no se clasifica con un umbral fijo en este notebook.
- **Valor sospechoso**: se aplica un umbral objetivo cuando existe un límite de negocio que permita justificarlo. En este caso, se consideran sospechosos los valores de `minimum_nights` superiores a 365 días, ya que no son coherentes con el concepto de alquiler turístico de corta estancia analizado en este proyecto.
- **Valor imposible**: no tiene interpretación coherente para el negocio.

Para `price` no existe un límite de negocio objetivo conocido (no hay un precio máximo "razonable" definido para alojamientos turísticos), así que se describe su distribución sin clasificar ningún valor concreto como sospechoso, la decisión se pospone a la fase de tratamiento con más contexto. Para `minimum_nights` sí se aplica un umbral explícito, como se muestra abajo.

No se elimina nada en esta fase, solo se documenta la evidencia. La decisión de tratamiento se toma más adelante, en el notebook de tratamiento y curación.

In [14]:
price_num = listings["price"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
price_num = pd.to_numeric(price_num, errors="coerce")

print("PRICE")
print(price_num.describe())
print("P95:", price_num.quantile(0.95), "| P99:", price_num.quantile(0.99))

print()
print("MINIMUM_NIGHTS")
print(listings["minimum_nights"].describe())
print("P95:", listings["minimum_nights"].quantile(0.95), "| P99:", listings["minimum_nights"].quantile(0.99))
print("Alojamientos con minimum_nights > 365 dias:", (listings["minimum_nights"] > 365).sum())

PRICE
count    13355.000000
mean       240.616470
std        339.474553
min          3.500000
25%         89.290000
50%        177.670000
75%        285.500000
max      10542.220000
Name: price, dtype: float64
P95: 548.0 | P99: 1815.0

MINIMUM_NIGHTS
count    15292.000000
mean        14.058920
std         21.359699
min          1.000000
25%          1.000000
50%          2.000000
75%         31.000000
max        900.000000
Name: minimum_nights, dtype: float64
P95: 32.0 | P99: 45.0
Alojamientos con minimum_nights > 365 dias: 3


**Interpretación**:

- **`price`**: mediana de 177,67€, P95 en 548€, P99 en 1.815€, máximo 10.542,22€. La distribución tiene una asimetría fuerte hacia la derecha, esperable en alojamientos turísticos (hay una diferencia real entre un piso compartido y una villa de lujo). No se considera automáticamente un error, un precio de varios miles de euros puede corresponder a un alojamiento real de gama alta. Se investigará su impacto sobre el modelo antes de decidir si se trata.
- **`minimum_nights`**: mediana de 2 noches, P95 en 32, P99 en 45, pero el máximo llega a 900. Tres alojamientos exigen más de 365 noches de estancia mínima, unos 2,5 años, un valor difícil de interpretar como una condición real de alquiler turístico de corta duración. Este sí se marca como **valor sospechoso que requiere investigación específica** en el tratamiento, a diferencia del precio. (Nota de completitud: `minimum_nights` tiene 1 valor nulo sobre 15.293, 0,0065%, muy por debajo del umbral del 10% definido en la Sección 1, no se reporta como hallazgo.)

## Resumen de hallazgos — tabla `listings`

**Problemas de calidad de datos**

| # | Hallazgo | Dimensión | Cifra |
|---|---|---|---|
| L1 | 12 columnas 100% vacías | Completitud | 12 columnas |
| L2 | Nulos parciales en price/bedrooms/bathrooms/beds/host_about/host_location | Completitud | 12,7%-31,8% según la columna |
| L3 | Campo `license` no verificable (sin dato + no clasificable) | Validez | 27,5% |
| L4 | Licencias con dos números concatenados + HTML sin limpiar | Validez | 5.429 registros |
| L5 | `minimum_nights` > 365 días, valor sospechoso | Coherencia | 3 alojamientos |

**Patrones de negocio derivados de los problemas anteriores** (no son problemas adicionales independientes, son desgloses de L3 y L2 que ayudan a priorizar dónde pesa más cada problema)

| # | Hallazgo | Deriva de | Cifra |
|---|---|---|---|
| L6 | Licencias no clasificables sobrerrepresentadas entre anfitriones profesionales | L3, por tipo de anfitrión | 54,9% vs 16-29% |
| L7 | Problema del campo `license` varía por distrito | L3, por distrito | 22,1% (Eixample) a 39,9% (Horta-Guinardó) |
| L8 | Completitud de `host_about`/`host_location` desigual por tipo de anfitrión | L2, por tipo de anfitrión | 11,0% vs 39,2% nulos; 6,8% vs 25,1% nulos |

**Verificaciones con resultado satisfactorio** (se comprobaron y no se detectaron problemas de calidad, por lo que no se incluyen en la tabla de hallazgos):

- Unicidad de `id` y ausencia de filas completamente duplicadas (Sección 2).
- Distribución de `price` (Sección 5): se observaron valores extremos, pero no existe un límite de negocio objetivo que permita clasificarlos como sospechosos.
- Codificación de caracteres en nombres de barrio (Sección 4): no se detectaron caracteres de reemplazo (�).

Estos 8 hallazgos (5 problemas de calidad + 3 patrones de negocio derivados) pasan a la tabla de recomendaciones de gobernanza, junto con los hallazgos identificados en `reviews` y `calendar`.